# AI Engineering Interview Prep
## Section: LangChain

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 651+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🦜 Section 15 — LangChain

### Q1. Why was LCEL (LangChain Expression Language) introduced, and what are its benefits?
**A:** LangChain initially used subclassed classes like `LLMChain` or `RetrievalQA`. They worked, but they were black boxes—hard to customize, trace, or modify. LCEL was introduced as a declarative syntax to compose chains from modular primitives (prompts, LLMs, parsers). Its major benefit is "features out of the box": any chain written in LCEL automatically supports synchronous execution (`invoke`), asynchronous (`ainvoke`), streaming (`stream` and `astream`), batching (`batch`), and parallel execution (using `RunnableParallel`). It gives you production-ready optimizations without writing custom threading or concurrency boilerplate.

### Q2. Explain the core components of the Runnable interface: Passthrough, Parallel, Lambda, and Sequence.
**A:** In LCEL, everything is a "Runnable" that implements a common interface.
- `RunnableSequence` (or the pipe `|` operator): Links runnables sequentially. The output of one becomes the input of the next (`prompt | model | parser`).
- `RunnableParallel`: Runs multiple runnables in parallel. Great for retrieving multiple retrievers at once or formatting independent dictionary keys.
- `RunnablePassthrough`: Passes the input through unchanged, or adds keys to the input dictionary. Essential for nesting retrieval results alongside the original query.
- `RunnableLambda`: Wraps any custom Python function in a Runnable, giving it access to the `invoke`, `stream`, and `async` methods.

### Q3. Why is LangChain deprecating legacy chains like LLMChain or RetrievalQA?
**A:** Legacy chains were rigid. If you wanted to stream intermediate tokens, run multiple chains in parallel, or inject custom logic mid-chain, you had to override internal class methods, which was fragile. Additionally, debuggability was poor because they didn't map cleanly to the execution graph. By deprecating them in favor of LCEL and LangGraph, LangChain forces a unified API. LCEL handles linear composition cleanly, and LangGraph handles cyclical/graph-based agents. It makes codebase maintenance easier and ensures features like token streaming work reliably.

### Q4. How does conversational memory work in modern LangChain?
**A:** Historically, we used `ConversationBufferMemory` passed directly into a chain. Now, LangChain uses `RunnableWithMessageHistory`. You define a session-based history store (e.g., using `ChatMessageHistory` or database-backed alternatives like Postgres) and wrap your LCEL chain with `RunnableWithMessageHistory`. When invoking, you pass a `session_id` in the `config`. The wrapper automatically retrieves the past messages for that session, injects them into the prompt placeholder, runs the chain, and updates the history with the new user input and AI response. It keeps state retrieval decoupled from the model logic.

### Q5. How do you stream tokens and intermediate steps in LangChain?
**A:** For basic token streaming, you just call `.stream()` or `.astream()` on the chain instead of `.invoke()`. But for complex chains where you need to stream the final answer *and* see which tools are running in the background, you use `.astream_events(version="v2")`. This returns an async event stream (e.g., `on_chat_model_stream` for tokens, `on_tool_start`/`on_tool_end` for tool calls). You filter these events in your event loop to update the UI dynamically, showing the user exactly what the agent is doing in real-time.

### Q6. How do you implement tool calling and ensure structured outputs in LangChain?
**A:** We use `.bind_tools()` to attach tool schemas (defined as Pydantic models or functions) to the LLM. This forces the model to return tool calls in a structured format rather than parsing text. To guarantee the final response matches a specific schema, we use `.with_structured_output(MyPydanticModel)`. Behind the scenes, LangChain attaches the schema, configures the provider's native JSON mode or function calling parameters, and automatically parses the output directly into a typed Pydantic object. It saves you from writing custom regex or JSON parsing logic.

### Q7. How does the LangChain callback system work, and how does it relate to LangSmith?
**A:** Callbacks are hooks triggered at key points in the pipeline (e.g., `on_llm_start`, `on_tool_end`). You can write custom callback handlers to log metrics, track tokens, or calculate costs. In production, the most common handler is the LangSmith integration. By simply setting `LANGCHAIN_TRACING_V2=true` in your environment, LangChain automatically registers a tracing callback that sends detailed execution logs to LangSmith. You get a visual run tree of every LLM call, prompt template, tool execution, and latency breakdown, which is invaluable for debugging.

### Q8. What are the best practices for document chunking and retrieval using LangChain?
**A:** Instead of basic character splitting, the modern standard is `RecursiveCharacterTextSplitter` which splits by paragraph, sentence, and word to keep semantic chunks together. For advanced RAG, we use hierarchical strategies like `ParentDocumentRetriever`. It splits documents into small chunks for vector search (which yields better similarity scores) but maps them back to larger "parent" chunks. When a small chunk is retrieved, LangChain fetches and feeds the larger parent chunk to the LLM. This ensures precise retrieval without starving the LLM of necessary context.

### Q9. How do you create custom tools in LangChain with validation?
**A:** The easiest way is using the `@tool` decorator on a Python function. The function's docstring acts as the tool's description for the LLM, and type hints are used to construct the tool's input schema. If you need complex validation, you subclass `BaseTool` and define an explicit Pydantic `args_schema`. LangChain automatically validates the LLM's arguments against this schema before execution. If the LLM generates bad parameters, the validator raises an error, which can be caught and sent back to the LLM for self-correction.

### Q10. How does vector store abstraction work in LangChain, and how do you configure custom retrievers?
**A:** LangChain provides a unified interface (`VectorStore`) that wraps databases like FAISS, Pinecone, or pgvector. You initialize the store, index your embeddings, and call `.as_retriever()`. You can customize retrieval by passing `search_type` and `search_kwargs`. For example, `search_type="mmr"` (Maximal Marginal Relevance) optimizes for a balance of relevance and diversity to avoid duplicate chunks. You can also pass metadata filters in `search_kwargs` to restrict the search to specific files or users, which is essential for multi-tenant applications.